# 08 — RAG: corpus, index and retrieval (contract)

**Owner:** RAG builder · **Status:** contract drafted, no code yet · **Last updated:** 2026-09-21

This notebook builds the retrieval layer that explains JobAI's vacancy forecasts using official Finnish sources. This first cell is the **contract**: what the RAG does, what it needs from the forecaster, and what it promises the evaluator. Read it before touching any RAG code. If you change something it describes, add a line to the change log at the bottom.

**Status labels used below**

| Label | Meaning |
|---|---|
| `agreed` | All owners confirmed it. Do not change it without telling the others. |
| `proposed` | Written from the existing repo. Needs confirmation from the named person. |
| `unverified` | Could not be checked. Do not rely on it. |

## 1. What the RAG does and does not do

The system answers questions such as *"What is the vacancy outlook for software developers in Uusimaa?"* in three steps:

```text
question ──► forecast record (from the fine-tuned model)
        └──► retrieve official passages (this notebook)
                         │
        forecast + history + passages ──► English answer with citations
```

| The RAG **owns** | The RAG **does not own** |
|---|---|
| Collecting and cleaning source documents | Fine-tuning or forecast quality (fine-tuner) |
| Chunking, embedding, the Chroma index | Retrieval and answer evaluation sets and metrics (evaluator) |
| The retrieval function | Numeric history (comes from PxWeb data) |
| Generating a cited answer | |

**Rules that always apply** (`agreed` unless noted)

1. **No numbers from retrieval.** Vacancy counts come from the PxWeb data and the forecast record. Retrieved text supplies context only.
2. **Every claim about causes needs a citation.** If no retrieved passage supports it, the answer says "no supporting source found" and does not invent a reason.
3. **Time cutoff.** Retrieval only returns documents published on or before the forecast origin date. This prevents later bulletins from leaking the outcome into an evaluation.
4. **Language.** Answers are in English. Finnish sources are kept in Finnish and not translated (`configs/rag.yaml`).

## 2. Input from the fine-tuner: the forecast record (`proposed`)

**Needs confirmation from: the fine-tuner.** This layout is inferred from notebooks 05, 06 and 07. The real forecast files (panel JSONL, prediction CSVs) are not in the repo, so field names have not been checked against real output.

The model itself returns `{"target_scaled_change": x}`. The RAG expects the converted record:

| Field | Type | Example | Notes |
|---|---|---|---|
| `series_id` | string | *(from the panel selection catalog)* | Identifies the series |
| `table_id` | string | `12tu` | `12tu` occupation×province, `12tw` industry×province |
| `dimensions` | object | `{"Alue": "MK01 Uusimaa", "Ammattiryhmä": "..."}` | Use the English `*_text` labels from `data/processed/*__normalized.csv` |
| `origin_quarter` | string | `2025Q4` | Also used as the retrieval date cutoff |
| `target_quarter` | string | `2026Q4` | |
| `horizon_q` | int | `1`, `2` or `4` | Only these horizons exist |
| `latest_value` | number | `36400` | Vacancy count at the origin quarter |
| `predicted_value` | number | `35100` | `latest_value + target_scaled_change × scale` |

**Open questions for the fine-tuner**

- Will the record arrive as a function call or as a file, and is the layout above right?
- Is the explainer the same model as the forecasting adapter? The RAG plan assumes a **separate** base/instruct model writes the explanation, because the adapter was trained to output numeric JSON only.

Until real records exist, the RAG is built against a stub record in this layout.

## 3. Output for the evaluator: what she can call (`proposed`)

**Needs confirmation from: the evaluator.**

### Retrieval

| | |
|---|---|
| **Input** | `query` (string), `k` (int, default 5), `before_date` (date, required) |
| **Output** | List of passages, best first |

Each passage has these fields:

| Field | Example | Use |
|---|---|---|
| `doc_id` | `keha_bulletin_2025-06` | Which document. Match this against expected sources |
| `chunk_id` | `keha_bulletin_2025-06#014` | Which passage inside it |
| `text` | *(passage text)* | What the generator sees |
| `score` | `0.71` | Retrieval similarity |
| `landing_url` | *(official URL)* | Link shown as the citation |
| `published_effective` | `2025-06-27` | Compared with `before_date`. Corrected date; never use raw `published` |
| `title` | `Työllisyyskatsaus, kesäkuu 2025` | Document title, shown in the citation |
| `heading` | `Avoimet työpaikat` | Section of the document (may be empty) |
| `language` | `fi` | `fi` or `en` |
| `source_type` | `keha_bulletin` | Which source family |

### Answer

| | |
|---|---|
| **Input** | `question`, one forecast record (section 2), `before_date` |
| **Output** | `answer` (English text), `cited_chunk_ids` (list), `supported` (false when nothing relevant was found) |

### Guarantees she can rely on

- Retrieval is **deterministic**: the same query and index always return the same passages.
- The index is **versioned**: its manifest (embedding model, chunking settings, document hashes) is saved under `data/manifests/`, following the existing manifest convention.
- `before_date` is always applied. No result has `published` after it.

### Suggested evaluation set format (evaluator's decision)

| Column | Purpose |
|---|---|
| `question` | English query. Include some about Finnish-only documents |
| `expected_doc_ids` | Which documents should be retrieved |
| `before_date` | Forecast origin date, to test the time cutoff |
| `language_of_source` | `fi` or `en` |
| `answerable` | Whether the corpus can answer it. Include some `false` cases |

Metrics in `configs/rag.yaml`: hit@5, MRR, nDCG, citation precision, citation completeness, faithfulness. Measure them **with and without** retrieval to show whether the RAG helps.

## 4. Source inventory

Checked on 2026-09-21 by reading the live pages. Ordered by reliability. **Version 1 of the corpus uses sources A and B.**

| ID | Source | What it gives | Coverage | Licence | Status |
|---|---|---|---|---|---|
| A | Statistics Finland job vacancy releases (`stat.fi/en/statistics/atp`) | Official facts about vacancy changes by region and industry. Does **not** explain causes | Quarterly. Only the 3 newest releases are reachable (list is JavaScript-rendered) | CC BY 4.0, covers text and tables. Attribution required | `agreed` as reliable, coverage `unverified` |
| B | KEHA Employment Bulletin (Työllisyyskatsaus) | Monthly context on unemployment, vacancies and services. Main source for explanations | 35 records Jan 2025 – Jul 2026 (English and Finnish) via DORIA, ~2 MB PDF each. Earlier issues on `tem.fi` and `tyollisyyskatsaus.fi`, reportedly 2013–2024, are not collected yet | **In Copyright** (DORIA rights field). Not open: raw files stay out of git | `proposed` (facts checked 2026-09-21) |
| C | TEM labour market forecast (Työmarkkinaennuste), `julkaisut.valtioneuvosto.fi` | Forward-looking drivers of labour demand | About twice a year, PDF | Not checked | `proposed`, **not yet in the `rag.yaml` whitelist** |
| D | Labour Force Barometer (`tyovoimabarometri.fi`) | Regional and sector labour demand and skills needs | Format, frequency and archive depth unknown | Unknown | `unverified` |

**Known limits**

- Sources A and B say little about **individual occupations**, so occupation-level questions will often get only general context and an "unsupported" answer. The evaluator should not count these as failures.
- **Whitelist:** `stat.fi/en/releases-and-publications` returned a 404 and was replaced by `stat.fi/en/statistics/atp` in `configs/rag.yaml`. Adding C or D needs a team decision.
- The collected bulletins start in Jan 2025, so **most of the test period (2024Q3–2026Q1) has little or no bulletin coverage yet**. Earlier bulletins must be added before evaluating that window.
- **Wrong dates in the source:** 2 DORIA records carry the wrong year. Notebook 09 corrects them into `published_effective`; the time cutoff must use that field, not `published`.

## 5. Plan for this notebook

1. Confirm the open questions in sections 2 and 3 with the fine-tuner and the evaluator.
2. Build the document inventory and download the files: done in notebook 09.
3. Clean and chunk (512 tokens, 64 overlap, per `configs/rag.yaml`): **done in notebook 10 (1,600 chunks)**.
4. Embed with `BAAI/bge-m3` and build the Chroma and BM25 indexes: **done in notebook 11**.
5. Implement and freeze the retrieval function (hybrid search, date filter, reranker off), then the answer function: **done in notebook 12**.
6. Write the rebuild instructions here for the evaluator: the frozen interface is documented at the top of notebook 12; a real (non-placeholder) answer generator is still needed.

## Change log

| Date | Who | Change |
|---|---|---|
| 2026-09-21 | RAG builder | Contract created from the existing repo and source research. Everything marked `proposed` or `unverified` is waiting for confirmation. |
| 2026-09-21 | RAG builder | Notebook 09 added. Findings: KEHA bulletins are In Copyright; 2 bulletin dates wrong in source; bulletin coverage starts Jan 2025; retrieval date field is `published_effective`. Fixed the 404 URL in `configs/rag.yaml`. |
| 2026-09-21 | RAG builder | Notebook 09 now covers 183 documents, every month 2013-01 to 2026-07 (Finnish; English from 2025-02). Notebook 10 added (clean + chunk, 1,600 chunks). Contract section 3 updated to the real chunk fields (`published_effective`, `landing_url`, `title`, `heading`). Decisions: hybrid search (semantic + BM25), reranker off in v1. Tables are deliberately not indexed. |
| 2026-09-22 | RAG builder | Notebook 11 added: embedded all 1,600 chunks with `bge-m3` (CPU, ~3.5 min) into ChromaDB, built a Finnish-stemmed BM25 index. Sanity checks found a real gap: the Finnish stemmer does not always match a query's base form to the document's inflected form (e.g. `pitkäaikaistyöttömyys`); semantic search still finds these, which supports keeping the search hybrid. Also found BM25's helper returns low/zero-score results as if they were matches -- notebook 12 must add a minimum-score cutoff. |
| 2026-09-22 | RAG builder | Notebook 12 added: `retrieve()` (hybrid RRF, date cutoff, per-method relevance floor, per-doc cap) and `answer()` (forecast stated from the record, cited passages, `supported=False` with no citations when nothing clears the floor) frozen against `STUB_FORECAST`. **`answer()`'s generator is a template, not an LLM call** -- pending the fine-tuner's confirmed forecast record and a decision on the explainer model. Fixed two bugs found while testing: `configs/rag.yaml`'s Chroma path did not match what notebook 11 wrote, and Chroma's metadata filter needs a number, not a date string, so a parallel `published_effective_epoch` integer field was added (notebook 11 re-run). Evaluator: the interface is stable, ready for the question set and metrics. |
